### Activation Functions

Sigmoid and softmax already have detailed treatment elsewhere (`classical-ml.ipynb`'s Logistic Regression section, and its numerically-stable-softmax appendix), referenced not repeated. This notebook is the comparison across the whole family, worked through the same input value so the vanishing-gradient contrast is directly visible.

#### 0. Why non-linear activations are needed at all

Stacking linear layers without a non-linearity between them collapses to a single linear layer, no matter how many layers deep: if layer 1 computes W1*x and layer 2 computes W2*(W1*x), that is just (W2*W1)*x, a single matrix multiplication, all the extra layers bought zero additional representational power. A non-linear activation between layers is what lets a deep network represent genuinely non-linear functions, without it, "deep" learning would be mathematically identical to a single linear model regardless of depth.

#### 1. Sigmoid and Tanh, worked at the same input to show saturation

Sigmoid: sigma(z) = 1/(1+e^-z), range (0,1), derivative = sigma(z)*(1-sigma(z)).
Tanh: range (-1,1), zero-centered (an advantage over sigmoid, whose outputs are always positive, which can bias gradient directions in later layers), derivative = 1 - tanh(z)^2.

Worked at z=3 (a moderately large positive input, not even extreme):
```
sigmoid(3) = 0.9526,  sigmoid'(3) = 0.9526*(1-0.9526) = 0.0452
tanh(3)    = 0.9951,  tanh'(3)    = 1 - 0.9951^2       = 0.0098
```
Both gradients are already small at z=3, and both continue shrinking toward 0 as z grows further, this is saturation, the exact mechanism behind the vanishing gradient problem covered in `nlp.ipynb`'s RNN section, repeated multiplication by numbers this small across many layers or timesteps drives the gradient toward zero. Tanh saturates even faster than sigmoid here (0.0098 vs 0.0452), despite being zero-centered.

In [ ]:
import numpy as np

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_grad(z):
    s = sigmoid(z)
    return s * (1 - s)

def tanh_grad(z):
    return 1 - np.tanh(z) ** 2

z = 3.0
print(f"sigmoid({z}) = {sigmoid(z):.4f}, gradient = {sigmoid_grad(z):.4f}")
print(f"tanh({z}) = {np.tanh(z):.4f}, gradient = {tanh_grad(z):.4f}")

#### 2. ReLU, and why it dominates deep networks

Formula: relu(z) = max(0, z). Derivative: 1 for z>0, 0 for z<0 (undefined exactly at 0, conventionally set to 0 or 1 by implementation).

Worked at the SAME z=3 used above:
```
relu(3) = 3
relu'(3) = 1     <- full gradient, no saturation at all for positive inputs
```
Compare to sigmoid's 0.0452 and tanh's 0.0098 at the identical input, ReLU's gradient does not shrink at all as z grows, for any positive input, the gradient is always exactly 1. This is the core reason ReLU became the default for deep networks, no saturation on the positive side means gradients can flow through many layers without vanishing, directly solving the problem sigmoid/tanh created.

In [ ]:
def relu(z):
    return np.maximum(0, z)

def relu_grad(z):
    return (z > 0).astype(float)

print(f"relu({z}) = {relu(z)}, gradient = {relu_grad(np.array(z))}")

#### 3. The dead ReLU problem, and Leaky ReLU's fix

ReLU's weakness is symmetric to its strength: for NEGATIVE inputs, both the output and the gradient are exactly 0.
```
relu(-2) = 0
relu'(-2) = 0    <- exactly zero, no gradient signal at all
```
If training pushes a neuron's weights such that its input is consistently negative across the whole dataset, that neuron outputs 0 for everything and receives a gradient of exactly 0 forever after, it is "dead," permanently inactive, with no mechanism to ever recover (a zero gradient means no weight update can happen through that path).

Leaky ReLU fixes this with a small non-zero slope for negative inputs instead of a hard zero: leaky_relu(z) = z if z>0, else alpha*z (alpha commonly 0.01).
```
leaky_relu(-2) = 0.01 * -2 = -0.02
leaky_relu'(-2) = 0.01           <- small, but NOT zero, the neuron can still receive a gradient signal and recover
```
ELU (Exponential Linear Unit) is a smoother variant of the same fix, using an exponential curve instead of a straight line for negative inputs, smoother gradients near the transition point at the cost of being more expensive to compute (involves exp()).

In [ ]:
def leaky_relu(z, alpha=0.01):
    return np.where(z > 0, z, alpha * z)

def leaky_relu_grad(z, alpha=0.01):
    return np.where(z > 0, 1.0, alpha)

z_neg = -2.0
print(f"relu({z_neg}) = {relu(z_neg)}, gradient = {relu_grad(np.array(z_neg))}")
print(f"leaky_relu({z_neg}) = {leaky_relu(z_neg):.3f}, gradient = {leaky_relu_grad(z_neg)}")

#### 4. GELU, briefly

Smooth approximation combining ReLU's shape with a probabilistic weighting (multiplies the input by the standard Normal CDF evaluated at that input, connecting back to `probability-statistics.ipynb`'s Normal distribution section), used as the default activation in most modern transformer architectures (see `llm-mechanics/llm-architecture.ipynb`), the smoothness (no hard kink at 0, unlike ReLU) tends to help optimization in very deep networks, at extra compute cost per activation compared to plain ReLU.

#### 5. Softmax, cross-referenced

Full mathematical treatment, the numerically-stable implementation, and worked multi-class examples already covered in `classical-ml.ipynb` (Logistic Regression section) and reused throughout `boosting.ipynb`'s multi-class XGBoost walkthrough. Not repeated here, the short version: softmax is the multi-class generalization of sigmoid, turning K raw scores into K probabilities that sum to 1, used as the FINAL layer activation for multi-class classification, never as a hidden-layer activation the way the functions above are.

In [ ]:
import torch
import torch.nn as nn

z_tensor = torch.tensor([3.0, -2.0, 0.5])

for name, fn in [("sigmoid", nn.Sigmoid()), ("tanh", nn.Tanh()), ("ReLU", nn.ReLU()),
                 ("LeakyReLU", nn.LeakyReLU(0.01)), ("ELU", nn.ELU()), ("GELU", nn.GELU())]:
    print(f"{name}({z_tensor.tolist()}) = {fn(z_tensor).tolist()}")